# Native Memory Pointer Pattern — Strands `ContextOffloader`

The companion notebook (`test_context_overflow.ipynb`) builds the Memory Pointer Pattern **by hand**: tools store large data in `agent.state` and return a pointer string. That teaches the concept from scratch.

This notebook shows the **same pattern as a first-class Strands feature**. Strands now ships `ContextOffloader` — a plugin that intercepts large tool results at execution time, stores each block in a storage backend, and leaves a small preview plus a reference in context. The offloading concern moves **out of your tools** and into the framework.

> This demo uses Strands Agents. Offloading large tool outputs and summarizing history are general agent concepts and carry over to other agent frameworks.

## Manual vs Native

| | Manual (`tools.py`) | Native (`native_tools.py`) |
|---|---|---|
| Fetch tool | Stores in `agent.state`, returns a pointer string | Ordinary function — just returns the JSON |
| Analysis tool | Receives `logs_pointer`, calls `agent.state.get()` | Ordinary function — no pointer logic |
| Who offloads | You, inside every tool | The `ContextOffloader` plugin, outside the tools |
| Retrieval | Read `agent.state` by key | `retrieve_offloaded_content(reference)` — by exact reference |

## What we test (same query, three strategies)

| Test | Strategy | What it shows |
|------|----------|---------------|
| 1 | No context management | Raw JSON enters the context window → high tokens |
| 2 | `ContextOffloader` (FileStorage) | Large results offloaded to disk → low tokens |
| 3 | `context_manager="auto"` | One line composes Summarizing + ContextOffloader |

## Setup: install dependencies

This demo needs **strands-agents 1.44.0+** (earlier versions don't have `ContextOffloader` or `context_manager="auto"`). Run the cell below once to install everything from `requirements.txt`, then restart the kernel if prompted.

In [ ]:
%pip install -r requirements.txt

# Verify the installed version is new enough for the native context APIs
import importlib.metadata as _m
_v = _m.version("strands-agents")
assert tuple(int(x) for x in _v.split(".")[:2]) >= (1, 44), (
    f"strands-agents {_v} is too old. This demo needs >= 1.44.0. "
    "Re-run the install cell and restart the kernel."
)
print(f"strands-agents {_v} — OK")

## Configure API Key

Set your OpenAI API key. Get one at https://platform.openai.com/api-keys

> You can swap to any provider supported by Strands — see [Strands Model Providers](https://strandsagents.com/docs/user-guide/concepts/model-providers/) for configuration.

In [ ]:
import os
# os.environ['OPENAI_API_KEY'] = 'your-key-here'  # Uncomment and set your key
assert os.getenv('OPENAI_API_KEY'), (
    '⚠️ OPENAI_API_KEY not set. '
    'Get yours at https://platform.openai.com/api-keys and set it above or in a .env file.'
)

## Setup

Note the imports: `native_tools` provides **ordinary** log tools (no pointer logic), and `ContextOffloader` / `FileStorage` come straight from Strands.

In [ ]:
import json, time, os, shutil

os.environ['OTEL_SDK_DISABLED'] = 'true'
import logging, warnings  # silence OpenTelemetry 'Failed to detach context' noise
logging.getLogger('opentelemetry').setLevel(logging.CRITICAL)
warnings.filterwarnings('ignore', message='Failed to detach context')

from dotenv import load_dotenv
from strands import Agent
# Using OpenAI-compatible interface via Strands SDK (not direct OpenAI usage)
from strands.models.openai import OpenAIModel
from strands.vended_plugins.context_offloader import ContextOffloader, FileStorage

from native_tools import fetch_application_logs, count_errors_by_service

load_dotenv()

MODEL = OpenAIModel(model_id='gpt-4o-mini')
ARTIFACT_DIR = './artifacts'

# Same query for all three tests — the only variable is the context-management strategy
QUERY = (
    "Fetch 2 hours of logs for 'api-gateway', then tell me how many errors occurred "
    'and which service had the most.'
)


def count_context_tokens(agent) -> int:
    """Approximate tokens across all messages in the conversation history (chars/4)."""
    total = 0
    for msg in agent.messages:
        content = msg.get('content', [])
        if isinstance(content, list):
            for block in content:
                if isinstance(block, dict):
                    if 'text' in block:
                        total += len(block['text']) // 4
                    elif 'toolResult' in block:
                        for item in block['toolResult'].get('content', []):
                            if 'text' in item:
                                total += len(item['text']) // 4
                    elif 'toolUse' in block:
                        total += len(json.dumps(block['toolUse'].get('input', {}))) // 4
    return total


# Start from a clean artifacts directory so Test 2's file count reflects this run only
if os.path.isdir(ARTIFACT_DIR):
    shutil.rmtree(ARTIFACT_DIR)

print('✅ Setup complete!')

---
## Test 1 — No Context Management (baseline)

Plain agent, plain tools. `fetch_application_logs` returns the full JSON dataset as a tool result, and it lands in the context window. Every subsequent model call re-sends it as input tokens.

In [ ]:
agent_baseline = Agent(model=MODEL, tools=[fetch_application_logs, count_errors_by_service])

start = time.time()
agent_baseline(QUERY)
time_baseline = time.time() - start
tokens_baseline = count_context_tokens(agent_baseline)

print(f'\n⏱️  {time_baseline:.1f}s')
print(f'📊 Tokens in context: {tokens_baseline:,}')

---
## Test 2 — `ContextOffloader` Plugin (native Memory Pointer Pattern)

Same tools, unchanged. We attach a `ContextOffloader` plugin backed by `FileStorage`. When a tool result exceeds `max_result_tokens`, the plugin stores it on disk and replaces it with a preview plus a reference — the raw data never stays in context.

Because `count_errors_by_service` is a **selective** tool (it computes the answer server-side and returns a small summary), the agent answers from the summary and the full logs stay offloaded. Offloader as the safety net, selective tools as the win.

In [ ]:
storage = FileStorage(artifact_dir=ARTIFACT_DIR)
agent_offload = Agent(
    model=MODEL,
    tools=[fetch_application_logs, count_errors_by_service],
    # Offload any tool result over ~800 tokens; keep a ~200-token preview in context
    plugins=[ContextOffloader(storage=storage, max_result_tokens=800, preview_tokens=200)],
)

start = time.time()
agent_offload(QUERY)
time_offload = time.time() - start
tokens_offload = count_context_tokens(agent_offload)

artifacts = [f for f in os.listdir(ARTIFACT_DIR) if not f.startswith('.')] if os.path.isdir(ARTIFACT_DIR) else []

print(f'\n⏱️  {time_offload:.1f}s')
print(f'📊 Tokens in context: {tokens_offload:,}')
print(f'📦 Artifacts offloaded to {ARTIFACT_DIR}/: {len(artifacts)} file(s) — retrievable by reference')

### Test 2b — Recover the data by its exact reference (the memory pointer)

The offloader put a short **reference** into the context in place of the data. That reference *is* the memory pointer. Here we pull it out of the conversation and use it to read the full dataset back from storage — byte for byte, by exact id. (The agent does this automatically via `retrieve_offloaded_content(reference)` when it needs the data; we do it by hand to make the pointer visible.)

In [ ]:
import asyncio, inspect, concurrent.futures

def extract_offload_reference(agent, artifact_dir):
    """Find the storage reference the ContextOffloader wrote into the conversation.

    The offloader replaces a large result with a preview that lists the stored
    references (here, FileStorage paths under artifact_dir). That reference is
    the memory pointer — a short string standing in for the full data.
    """
    for msg in agent.messages:
        for block in msg.get('content', []):
            if isinstance(block, dict) and 'toolResult' in block:
                for item in block['toolResult'].get('content', []):
                    for token in item.get('text', '').split():
                        if artifact_dir.strip('./') in token and token.endswith(('.txt', '.json')):
                            return token
    return None

def storage_retrieve(storage, reference):
    """Read content back, handling sync (<=1.43) and async (1.44+) APIs.
    Also works inside Jupyter's running event loop, where asyncio.run() would raise."""
    result = storage.retrieve(reference)
    if not inspect.isawaitable(result):
        return result
    try:
        asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(result)
    with concurrent.futures.ThreadPoolExecutor(1) as ex:
        return ex.submit(asyncio.run, result).result()

reference = extract_offload_reference(agent_offload, ARTIFACT_DIR)
print(f'Pointer in context: {reference}')

# Read the full dataset back by its exact reference — no loss
data_bytes, content_type = storage_retrieve(storage, reference)
events = json.loads(data_bytes)
print(f'storage.retrieve() → {len(data_bytes):,} bytes ({content_type})')
print(f'✅ {len(events):,} log events recovered verbatim by exact reference — never re-entered the context window')

---
## Test 3 — `context_manager="auto"` (one-line setup)

For most multi-turn agents you do not need to wire up offloading and summarization separately. Passing `context_manager="auto"` composes, with benchmark-validated defaults:

- `SummarizingConversationManager` (summarizes old history instead of dropping it, with proactive compression)
- `ContextOffloader` (in-memory storage) for large tool results

Your own `conversation_manager` or `plugins`, if provided, take precedence.

In [ ]:
agent_auto = Agent(
    model=MODEL,
    tools=[fetch_application_logs, count_errors_by_service],
    context_manager='auto',
)

start = time.time()
agent_auto(QUERY)
time_auto = time.time() - start
tokens_auto = count_context_tokens(agent_auto)

print(f'\n⏱️  {time_auto:.1f}s')
print(f'📊 Tokens in context: {tokens_auto:,}')
print(f'⚙️  Composed: {type(agent_auto.conversation_manager).__name__} + ContextOffloader (in-memory)')

---
## Comparison

In [ ]:
rows = [
    ('1 — No management', tokens_baseline, time_baseline),
    ('2 — ContextOffloader', tokens_offload, time_offload),
    ('3 — context_manager=auto', tokens_auto, time_auto),
]

print(f"{'Strategy':<32} {'Tokens':>10} {'Time':>8}")
print('-' * 52)
for label, tokens, elapsed in rows:
    print(f'{label:<32} {tokens:>10,} {elapsed:>6.1f}s')

best_label, best_tokens, _ = min(rows[1:], key=lambda r: r[1])
if tokens_baseline > best_tokens > 0:
    reduction = (1 - best_tokens / tokens_baseline) * 100
    print(f'\n→ Best native strategy: {best_label} — {reduction:.0f}% fewer tokens than baseline')

---
## Summary

### Manual vs Native — when to use which

- **Manual (`agent.state`)** — when you want full control over what is stored and how it is keyed, or you are teaching the pattern from first principles.
- **`ContextOffloader`** — when you want offloading applied automatically to *any* large tool result without changing tool code. Swap `FileStorage` for `S3Storage` to persist across machines.
- **`context_manager="auto"`** — the one-line default for most multi-turn agents; combines summarization and offloading.

### Taking it to production: two kinds of memory

`agentcore_production.py` shows the production split — and it is a deliberate one:

| Memory type | Backend | Retrieval |
|-------------|---------|-----------|
| **Conversation** (turns, preferences, facts) | AgentCore Memory | Semantic (similarity) |
| **Context data** (large tool outputs) | S3 (via `ContextOffloader(S3Storage(...))`) | Exact reference |

Logs and datasets need to come back **verbatim, by exact id** — that is object storage (S3), not the semantic recall of conversational memory. Keeping the two separate is what makes the production design clean.

### References

- [Strands Context Management](https://strandsagents.com/docs/user-guide/concepts/context-management/)
- [Strands Conversation Management](https://strandsagents.com/docs/user-guide/concepts/agents/conversation-management/)
- [IBM Research: Solving Context Window Overflow in AI Agents](https://arxiv.org/html/2511.22729v1)
- [Code Repository](https://github.com/aws-samples/sample-why-agents-fail)